# MCMP Object Storage Data Lab
위에서부터 셀을 실행하면 **버킷 목록 → 파일 선택 → 표 미리보기 → 집계·차트**를 볼 수 있습니다.
기본값은 허용된 첫 버킷에서 작은 CSV 파일 하나(CSV가 없으면 Parquet)를 선택합니다. 모든 파일을 합치지는 않습니다.
파일/컬럼 설정을 바꾸고 아래 셀을 다시 실행하세요. 자동으로 선택한 집계 컬럼이 분석 목적에 맞는지 확인하세요.

CSP 자격증명은 VM에 저장하지 않습니다. AM에서 필요할 때 발급받는 Presigned URL로 파일을 읽습니다.
이 노트북은 버킷을 마운트하지 않으며, 전체 셀을 실행해도 버킷에 업로드하거나 원본을 수정하지 않습니다.
다운로드는 기본 25 MiB로 제한합니다. 압축된 Parquet를 읽을 때 실제 메모리 사용량은 더 커질 수 있습니다.

In [ ]:
import io
import os
from pathlib import Path
import pandas as pd
import requests

gateway = os.environ['MCMP_OBJECT_STORAGE_GATEWAY_URL'].rstrip('/')
access_token = os.environ['MCMP_OBJECT_STORAGE_TOKEN']
auth_headers = {'Authorization': 'Bearer ' + access_token}

def _mcmp(method, path, **kwargs):
    response = requests.request(method, gateway + path, headers=auth_headers, timeout=30, **kwargs)
    response.raise_for_status()
    payload = response.json()
    if payload.get('code') != 200:
        raise RuntimeError(payload.get('detail') or payload.get('message') or 'Application Manager request failed')
    return payload.get('data')

def storages():
    return _mcmp('GET', '/storages')

def list_objects(storage, prefix=''):
    data = _mcmp('GET', '/objects', params={'storage': storage, 'prefix': prefix})
    return pd.DataFrame(data.get('objects', []))

def _presigned(storage, object_key, operation):
    return _mcmp('POST', '/presigned-url', json={'storage': storage, 'objectKey': object_key, 'operation': operation})

def download(storage, object_key, destination=None):
    ticket = _presigned(storage, object_key, 'download')
    response = requests.request(ticket['method'], ticket['presignedURL'], headers=ticket.get('requiredHeaders') or {}, timeout=300)
    response.raise_for_status()
    if destination is None:
        return response.content
    Path(destination).write_bytes(response.content)
    return Path(destination)

def upload(storage, source, object_key):
    ticket = _presigned(storage, object_key, 'upload')
    with Path(source).open('rb') as stream:
        response = requests.request(ticket['method'], ticket['presignedURL'], headers=ticket.get('requiredHeaders') or {}, data=stream, timeout=300)
    response.raise_for_status()
    return object_key

def read_csv(storage, object_key, **kwargs):
    return pd.read_csv(io.BytesIO(download(storage, object_key)), **kwargs)

print('Object Storage helpers are ready.')

In [ ]:
from IPython.display import display

def _preview_bytes(storage, object_key, limit_bytes):
    """Bound the preview download without printing presigned URLs in errors."""
    ticket = _presigned(storage, object_key, 'download')
    try:
        with requests.request(ticket['method'], ticket['presignedURL'],
                              headers=ticket.get('requiredHeaders') or {},
                              stream=True, timeout=60) as response:
            if not response.ok:
                raise RuntimeError(f"Object download failed (HTTP {response.status_code}).")
            chunks, total = [], 0
            for chunk in response.iter_content(chunk_size=65536):
                total += len(chunk)
                if total > limit_bytes:
                    raise ValueError('File exceeds MAX_DOWNLOAD_MB. Choose a smaller file or raise the limit.')
                chunks.append(chunk)
            return b''.join(chunks)
    except requests.RequestException:
        raise RuntimeError('Object download failed. Check connectivity and retry to obtain a fresh URL.') from None

def _choose_preview_key(items, requested_key, limit_bytes):
    supported = [o for o in items if str(o.get('key', '')).lower().endswith(('.csv', '.parquet'))]
    if requested_key:
        selected = next((o for o in supported if o['key'] == requested_key), None)
        if selected is None:
            raise ValueError('OBJECT_KEY must be a listed CSV or Parquet file within the allowed prefix.')
        if int(selected.get('size') or 0) > limit_bytes:
            raise ValueError('Selected file exceeds MAX_DOWNLOAD_MB.')
        return requested_key
    supported = [o for o in supported if not any(part.startswith('.') for part in o['key'].split('/'))]
    supported.sort(key=lambda o: (not o['key'].lower().endswith('.csv'), o['key']))
    return next((o['key'] for o in supported if int(o.get('size') or 0) <= limit_bytes), None)

def _summarize(data, group_col=None, value_col=None, aggregation='sum'):
    if data is None or data.empty:
        return None, group_col, value_col
    if aggregation not in {'sum', 'mean', 'min', 'max', 'count'}:
        raise ValueError('AGGREGATION must be sum, mean, min, max or count.')
    text_cols = list(data.select_dtypes(include=['object', 'string', 'category']).columns)
    num_cols = list(data.select_dtypes(include='number').columns)
    group_col = group_col or ('region' if 'region' in data.columns else (text_cols[0] if text_cols else None))
    value_col = value_col or (num_cols[0] if num_cols else None)
    if group_col is None or value_col is None:
        return None, group_col, value_col
    if group_col not in data.columns or value_col not in data.columns:
        raise ValueError('GROUP_COL and VALUE_COL must name columns in the loaded table.')
    if value_col not in num_cols:
        raise ValueError('VALUE_COL must be numeric. Check the CSV format or choose another column.')
    clean = pd.DataFrame({
        'group_key': data[group_col].astype('string').fillna('(missing)'),
        'value': pd.to_numeric(data[value_col], errors='coerce')
    })
    clean = clean[clean['value'].notna() & ~clean['value'].isin([float('inf'), float('-inf')])]
    if clean.empty:
        return None, group_col, value_col
    summary = clean.groupby('group_key', dropna=False)['value'].agg(aggregation).reset_index()
    return summary.sort_values('value', ascending=False), group_col, value_col

## 1. 연결된 버킷
배포 시 선택한 버킷과 접근 권한만 표시됩니다.

In [ ]:
available_storages = []
available_storages = storages() or []
display(pd.DataFrame(available_storages))
if not available_storages:
    print('No granted storage found. Check the Object Storage selection in Application Manager.')

## 2. 버킷·파일 선택
`STORAGE_ALIAS`와 `OBJECT_KEY`가 None이면 자동 선택합니다.
`OBJECT_PREFIX`는 버킷 기준 전체 접두사입니다. 예: `sample-data/`. 빈 문자열이면 배포 시 허용된 접두사를 사용합니다.
특정 파일 예: `OBJECT_KEY = 'sample-data/csv/usage-2026-06.csv'`.
설정을 변경하면 **3번부터 다시 실행**하세요.

In [ ]:
STORAGE_ALIAS = None
OBJECT_PREFIX = ''
OBJECT_KEY = None
MAX_DOWNLOAD_MB = 25
PREVIEW_ROWS = 10

## 3. 버킷 파일 목록
목록은 화면에 최대 50개를 표시합니다. 버킷에 새 파일을 올리면 이 셀부터 다시 실행하세요.

In [ ]:
df, summary, selected_key, selected_storage = None, None, None, None
objects = pd.DataFrame()
if not available_storages:
    print('Run section 1 and check the granted storages.')
else:
    alias = STORAGE_ALIAS or available_storages[0]['alias']
    selected_storage = next((s for s in available_storages if s['alias'] == alias), None)
    if selected_storage is None:
        raise ValueError('STORAGE_ALIAS must match an alias shown in section 1.')
    objects = list_objects(alias, OBJECT_PREFIX)
    print(f"Storage: {alias} | access: {selected_storage.get('accessMode')} | listed objects: {len(objects)}")
    display(objects.head(50))
    if objects.empty:
        print('No objects found. Check the prefix or upload data to the bucket, then rerun this cell.')

## 4. 데이터 미리보기
CSV 또는 Parquet 파일 **하나**를 읽습니다. 숨김 파일·폴더는 자동 선택하지 않습니다.
다운로드 크기를 넘는 파일은 자동 선택에서 제외합니다. 큰 데이터는 별도의 분석 작업으로 처리하세요.
Parquet에는 pyarrow 또는 fastparquet가 필요합니다. CSV 옵션이 필요하면 아래 `CSV_OPTIONS`를 수정하세요.

In [ ]:
df, summary, selected_key = None, None, None
CSV_OPTIONS = {}  # Example: {'encoding': 'utf-8', 'sep': ','}
if MAX_DOWNLOAD_MB <= 0 or PREVIEW_ROWS <= 0:
    raise ValueError('MAX_DOWNLOAD_MB and PREVIEW_ROWS must be positive.')
if selected_storage is None or objects.empty:
    print('No files to preview. Run section 3 first.')
else:
    limit_bytes = int(MAX_DOWNLOAD_MB * 1024 * 1024)
    selected_key = _choose_preview_key(objects.to_dict('records'), OBJECT_KEY, limit_bytes)
    if selected_key is None:
        print('No CSV/Parquet file within the download limit. Check the prefix, file type or MAX_DOWNLOAD_MB.')
    else:
        payload = _preview_bytes(selected_storage['alias'], selected_key, limit_bytes)
        try:
            if selected_key.lower().endswith('.parquet'):
                df = pd.read_parquet(io.BytesIO(payload))
            else:
                df = pd.read_csv(io.BytesIO(payload), **CSV_OPTIONS)
        except ImportError:
            print('Parquet support is missing. Use CSV or an image with pyarrow/fastparquet installed.')
        except pd.errors.EmptyDataError:
            print('The selected CSV contains no data.')
        finally:
            del payload
        if df is not None:
            print(f"File: {selected_key} | rows: {len(df)} | columns: {len(df.columns)}")
            display(df.head(PREVIEW_ROWS))
            display(df.dtypes.astype(str).rename('dtype').to_frame())

## 5. 집계와 차트
그룹 컬럼은 기본적으로 `region` 또는 첫 문자열 컬럼, 값 컬럼은 첫 숫자 컬럼입니다.
**자동 선택된 컬럼과 단위를 반드시 확인**하고 원하는 `GROUP_COL`, `VALUE_COL`을 지정하세요.
`AGGREGATION`: sum / mean / min / max / count. count는 선택한 값 컬럼의 유효한 값 개수입니다.
선택한 파일 하나에 대한 집계이며, 차트는 집계값이 큰 그룹부터 `TOP_N`개를 표시합니다.

In [ ]:
import matplotlib.pyplot as plt

GROUP_COL = None
VALUE_COL = None
AGGREGATION = 'sum'
TOP_N = 20

summary = None
if TOP_N <= 0:
    raise ValueError('TOP_N must be positive.')
summary, group_col, value_col = _summarize(df, GROUP_COL, VALUE_COL, AGGREGATION)
if df is None or df.empty:
    print('No loaded data. Run section 4 first.')
elif summary is None:
    print('No suitable grouping/numeric columns or finite values. Set GROUP_COL and VALUE_COL explicitly.')
else:
    print(f"Group: {group_col} | value: {value_col} | aggregation: {AGGREGATION}")
    display(summary.head(TOP_N))
    fig, ax = plt.subplots(figsize=(12, 5))
    summary.head(TOP_N).plot.bar(x='group_key', y='value', legend=False, ax=ax)
    ax.set_xlabel(str(group_col))
    ax.set_ylabel(f"{AGGREGATION}({value_col})")
    ax.set_title(f"{value_col} by {group_col} — top {min(TOP_N, len(summary))} groups")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 6. 결과 저장 (선택)
기본 실행은 읽기 전용입니다. 아래 예제는 **주석을 해제할 때만** 실행됩니다.
로컬 CSV 저장은 VM 작업 폴더에 저장합니다. 버킷 업로드는 READ_WRITE 권한과 허용된 경로가 필요합니다.
원본을 덮어쓰지 않도록 결과 파일 경로를 확인하세요.

In [ ]:
# if summary is not None:
#     summary.to_csv('summary.csv', index=False)  # Local VM file only.
#     upload(selected_storage['alias'], 'summary.csv', 'results/summary.csv')